### Задания 1-3 
В этой серии заданий вам понадобится сделать эмбеддинги для объектов из некоторой подвыборки датасета IMDB с помощью предобученных моделей из Hugging Face. 
1. С помощью модели BERT (bert-base-cased)
2. С помощью модели RoBERTa (roberta-base)
3. С помощью модели DistilBERT (distilbert-base-cased)

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import sys
import transformers
from transformers import AutoTokenizer
from transformers import BertModel
from transformers import RobertaModel
from transformers import DistilBertModel
from datasets import load_dataset
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
from torch.utils.data import Subset



from warnings import filterwarnings

filterwarnings("ignore")

C:\Users\olegs\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
dataset = load_dataset("imdb", split="train")

np.random.seed(100)

idx = np.random.randint(len(dataset), size=200)

In [3]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [4]:
def get_model(model_name):
    assert model_name in ['bert', 'roberta', 'distilbert']
    
    checkpoint_names = {
        'bert': 'bert-base-cased',
        'roberta': 'roberta-base',
        'distilbert': 'distilbert-base-cased'
    }
    
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_names[model_name])
    
    if model_name == 'roberta':
        model = RobertaModel.from_pretrained(
            checkpoint_names[model_name],
            add_pooling_layer=False
        )
    elif model_name == 'bert':
        model = BertModel.from_pretrained(checkpoint_names[model_name])
    else:
        model = DistilBertModel.from_pretrained(checkpoint_names[model_name])
        
    return tokenizer, model

In [13]:
tokenizer, model = get_model('bert')

In [14]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)
print(torch.cuda.get_device_name())

cuda:0
NVIDIA GeForce RTX 2060


In [15]:
model = model.to(device)

In [16]:
def tokenization(example):
    return tokenizer.batch_encode_plus(example['text'], add_special_tokens=True, return_token_type_ids=False, truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataset = dataset.map(tokenization, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(Subset(train_dataset, idx), batch_size=64, collate_fn=data_collator, pin_memory=True, shuffle=False)

In [17]:
from tqdm import tqdm


@torch.inference_mode()
def get_embeddings_labels(model, loader):
    model.eval()
    
    total_embeddings = []
    labels = []
    
    for batch in tqdm(loader):
        labels.append(batch['labels'].unsqueeze(1))

        batch = {key: batch[key].to(device) for key in ['attention_mask', 'input_ids']}

        embeddings = model(**batch)['last_hidden_state'][:, 0, :]

        total_embeddings.append(embeddings.cpu())

    return torch.cat(total_embeddings, dim=0), torch.cat(labels, dim=0).to(torch.float32)

In [18]:
train_embeddings, train_labels = get_embeddings_labels(model, train_loader)

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:04<00:00,  1.12s/it]


In [11]:
torch.save(train_embeddings, "roberta_embeddings.pt")

In [12]:
train_embeddings.shape

torch.Size([200, 768])

In [12]:
torch.cuda.empty_cache()

In [19]:
for name in ['roberta', 'bert', 'distilbert']:

    tokenizer, model = get_model(name)

    model = model.to(device)
    
    train_embeddings, train_labels = get_embeddings_labels(model, train_loader)

    torch.save(train_embeddings, f"{name}_embeddings.pt")

100%|████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.76it/s]
